# AI Agents on the CLI: The Coding-Agent Landscape

This notebook is the **landscape / vendor-comparison entry point** for Phase 11 (`11_Claude_Code_and_AI_Coding_Tools/`). It surveys the category of agentic, terminal-native coding tools — Claude Code and its peers — explains the mechanics they all share, and then builds a tiny toy CLI coding agent from scratch so the loop stops being an abstraction and becomes something you've actually run.

This is **not** a Claude Code tutorial and **not** an Agent SDK tutorial. Those live in this same phase's sibling tracks:

- `Claude_Code/` — Claude Code operator mastery (using the real product day to day)
- `Claude_API_and_Agent_SDK/` — building your own production agent on the Claude API / Agent SDK
- `AI_Coding_Tool_Landscape/` (this notebook) — the wider category, why it exists, and how the loop works underneath any vendor's implementation

## What we are going to do

1. Define the category: what makes a **CLI coding agent** different from a chat-based coding assistant.
2. Survey the landscape as of today: Claude Code and comparable tools.
3. Walk through the agentic loop common to all of them, with a diagram.
4. Build a **minimal toy CLI coding agent from scratch** (using `helpers.get_llm`, not any real product) that reads a scratch Python file, asks an LLM to edit it per a natural-language instruction, applies the edit, and shows a diff.
5. Discuss the permission/autonomy spectrum real tools expose, and why it matters.
6. Wrap up with where this fits relative to the rest of Phase 11.

## 1. The Landscape: What Is a "CLI Coding Agent"?

For years, AI coding assistance meant a **chat panel next to your editor**: you pasted code in, the model suggested code back, and you copy-pasted it into your files yourself. The model never touched your filesystem, never ran your tests, never saw the result of its own suggestion unless you told it.

A **CLI coding agent** changes the shape of that interaction in three ways:

| Property | Chat-based assistant | CLI coding agent |
|---|---|---|
| **Filesystem access** | None — you copy/paste | Direct — it reads and edits files itself |
| **Shell access** | None | Direct — it runs commands, tests, builds, linters |
| **Execution model** | Single turn: prompt in, suggestion out | A **loop**: it acts, observes the result, and keeps going until the task is done |
| **Autonomy** | Fully manual (you decide what to apply) | Configurable — from "ask before every write" to "run unattended" |

The defining trait is the **loop**. A chat assistant produces one artifact per turn and stops. A CLI coding agent keeps a working session open on your real repository: it can read a file, propose an edit, apply it, run your test suite, see the failure output, and try again — without you re-pasting anything in between.

### The landscape today (2026)

This is a survey, not an endorsement of any single tool — the point is the *shape* of the category, since new entrants and version changes are constant.

| Tool | Vendor | Interface | Notable characteristic |
|---|---|---|---|
| **Claude Code** | Anthropic | Terminal-native CLI + SDK | Deep tool-use loop, subagents, hooks, MCP support, plan mode; this repo's primary agent (see `Claude_Code/`) |
| **OpenAI Codex CLI** | OpenAI | Terminal-native CLI | Sandbox-first execution, approval modes tied to sandbox trust level |
| **GitHub Copilot CLI / agent mode** | GitHub / Microsoft | CLI + editor-integrated agent mode | Tight GitHub integration (issues, PRs, Actions) |
| **Cursor (Agent mode)** | Cursor / Anysphere | IDE-embedded, but drives shell + multi-file edits like a CLI agent would | Blurs the line between "editor assistant" and "agent" by giving the in-editor agent shell + multi-file powers |
| **Aider** | Open source | Terminal-native CLI | Early, influential pure-CLI pair-programmer; git-commit-per-edit workflow |
| **Amazon Q Developer / others** | Various | CLI and IDE variants | Same category, cloud-vendor-specific integrations |

What unites all of them is not the vendor or the model — it's that each exposes the **same underlying loop** (Section 2) and the same **permission spectrum** (Section 4). Once you understand those two things, evaluating a new entrant is mostly a matter of asking "how does *this* tool implement the loop, and how does it gate autonomy?"

## 2. Core Mechanics: The Agentic Loop

Strip away branding and every CLI coding agent reduces to the same control loop. The names differ by product, but the stages don't:

```mermaid
flowchart TD
    A["User gives a natural-language task\ne.g. 'add type hints to this function'"] --> B["Context gathering\nread relevant files, search the repo,\ninspect project structure"]
    B --> C["Planning\ndecide what changes are needed\nand in what order"]
    C --> D{"Permission gate?\n(edit / shell command\nabout to run)"}
    D -- "auto-approved\n(e.g. read-only)" --> E
    D -- "needs approval" --> G["Prompt human\nApprove / deny / edit"]
    G -- approved --> E
    G -- denied --> C
    E["Tool call\nedit file / run shell command /\nrun tests / run linter"] --> F["Observation\ncapture stdout, stderr,\nexit code, diff"]
    F --> H{"Task complete or\nblocked?"}
    H -- "no, iterate" --> B
    H -- "yes, done" --> I["Report result to user"]
    H -- "blocked, needs input" --> G
```

In prose:

1. **Context gathering** — before proposing anything, the agent reads the files it thinks are relevant (often via search/grep-style tools rather than reading the whole repo blindly).
2. **Planning** — the model reasons about what change satisfies the task, sometimes explicitly (a visible "plan" step), sometimes implicitly inside a single completion.
3. **Tool calls** — the agent doesn't emit code as text for a human to paste; it calls structured tools: `edit_file`, `run_shell_command`, `run_tests`, etc.
4. **Observation** — the *result* of each tool call (file diff, stdout/stderr, exit code) is fed back into the model's context. This is the step chat assistants fundamentally lack: the agent sees whether its own change actually worked.
5. **Repeat** — steps 1–4 loop until the model decides the task is done, or a **permission gate** stops it and asks a human to approve, deny, or redirect.

That feedback loop ("observe the result of your own action, then decide the next action") is the single mechanical idea that separates an *agent* from an *assistant*. Everything else — which model, which tools, which permissions UI — is implementation detail on top of this loop.

## 3. Hands-On: A Minimal Toy CLI Coding Agent From Scratch

### What we are going to do

To make the loop in Section 2 concrete, we build a deliberately tiny coding agent — **not** a wrapper around Claude Code, Codex, Aider, or any other real product. It:

1. Creates a small scratch Python file with `tempfile` (our stand-in "repo").
2. Takes a natural-language task (e.g. *"add type hints to this function"*).
3. **Context gathering**: reads the file's current contents.
4. **Planning + tool call (edit)**: sends the file contents and task to an LLM via `from helpers import get_llm`, asking it to return the full corrected file.
5. **Apply the edit**: writes the model's output back to the scratch file.
6. **Observation**: computes and prints a unified diff so we can see exactly what changed — the same feedback a real agent would fold back into its next loop iteration (e.g. before deciding whether to run tests next).

This intentionally has no test runner, no multi-file awareness, and no retry logic — real tools add all of that. The goal here is only to make the *edit-a-file-under-agent-control* primitive tangible and runnable.

In [ ]:
# ============ IMPORTS ============
import difflib
import tempfile
from pathlib import Path

from helpers import get_llm

In [ ]:
# ============ STEP 1: CREATE A SCRATCH FILE (our toy "repo") ============
scratch_source = '''def add(a, b):
    return a + b


def greet(name):
    return "Hello, " + name
'''

scratch_dir = Path(tempfile.mkdtemp(prefix="toy_cli_agent_"))
scratch_file = scratch_dir / "scratch_module.py"
scratch_file.write_text(scratch_source, encoding="utf-8")

print(f"Scratch file created at: {scratch_file}")
print("--- original contents ---")
print(scratch_source)

In [ ]:
# ============ STEP 2: THE TOY AGENT LOOP ============
# A minimal version of the loop from Section 2:
#   context gathering -> planning/tool call (edit) -> apply -> observation (diff)


def context_gathering(file_path: Path) -> str:
    """Read the target file's current contents (the agent's 'context')."""
    return file_path.read_text(encoding="utf-8")


def plan_and_edit(task: str, current_contents: str) -> str:
    """Ask the LLM to plan the change and return the full edited file.

    This is the 'tool call: edit file' step of the loop, implemented as a
    single LLM completion that must return the complete new file contents
    (no explanations, no markdown fences) so we can apply it directly.
    """
    llm = get_llm(temperature=0)

    system_prompt = (
        "You are a minimal CLI coding agent. You will be given the full "
        "contents of a Python file and a task. Return ONLY the complete, "
        "corrected file contents that satisfy the task. Do not include "
        "markdown code fences, explanations, or commentary -- your entire "
        "response must be valid Python source code for the whole file."
    )
    user_prompt = (
        f"Task: {task}\n\n"
        f"Current file contents:\n{current_contents}"
    )

    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    proposed = response.content.strip()

    # Defensive cleanup in case the model wraps the answer in a code fence
    # despite instructions -- real agents budget for imperfect tool output too.
    if proposed.startswith("```"):
        lines = proposed.splitlines()
        lines = lines[1:] if lines[0].startswith("```") else lines
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        proposed = "\n".join(lines) + "\n"

    return proposed


def apply_edit(file_path: Path, new_contents: str) -> None:
    """Write the model's proposed contents back to disk (the 'apply' step)."""
    file_path.write_text(new_contents, encoding="utf-8")


def observe_diff(before: str, after: str, file_name: str) -> str:
    """Produce a unified diff -- the 'observation' the agent (or a human
    reviewer) would use to decide whether the task is complete."""
    diff_lines = difflib.unified_diff(
        before.splitlines(keepends=True),
        after.splitlines(keepends=True),
        fromfile=f"a/{file_name}",
        tofile=f"b/{file_name}",
    )
    return "".join(diff_lines)


def run_toy_cli_agent(task: str, file_path: Path) -> str:
    """Run one iteration of the toy agentic loop end to end."""
    before = context_gathering(file_path)
    after = plan_and_edit(task, before)
    apply_edit(file_path, after)
    return observe_diff(before, after, file_path.name)


print("Toy agent defined. Ready to run.")

In [ ]:
# ============ STEP 3: RUN THE LOOP ON A NATURAL-LANGUAGE TASK ============
task = "Add type hints to every function in this file."

diff_output = run_toy_cli_agent(task, scratch_file)

print("--- diff observed after applying the edit ---")
print(diff_output if diff_output else "(no changes produced)")

### Discussion of the Output

The diff printed above is the same artifact a real CLI coding agent surfaces at the end of an edit step — and it's exactly what would get fed back into the loop's next "observation" stage in a multi-step task (e.g. "now run `mypy` on the file and fix any new errors it reports").

Notice what this toy agent does **not** do, which is exactly what separates it from a real product:

- It doesn't re-read the file after editing to verify the change actually satisfies the task (no self-check / reflection step).
- It doesn't run anything (no test suite, no linter, no type checker) to validate the edit — a real agent would run `mypy` here and loop again if it failed.
- It applies the edit unconditionally — there's no permission gate. Section 4 covers why real tools almost never do this by default for anything beyond trivial changes.
- It handles exactly one file and one edit — no multi-file reasoning, no planning across steps.

Every one of those gaps is a real feature in production CLI agents (Claude Code included) — this toy exists purely to make the *edit-under-agent-control-and-observe-the-diff* primitive something you've run yourself, not just read about.

## 4. Permission and Autonomy Models

The toy agent above applied its edit unconditionally — no human was asked. Real CLI coding agents almost never default to that for anything beyond the safest actions, because an agent with filesystem and shell access can do real damage from a single bad tool call (deleting files, force-pushing, leaking secrets via a curl command, running `rm -rf`, etc.).

Every mainstream tool in the landscape exposes some version of the same **autonomy spectrum**:

| Level | Typical scope | Example |
|---|---|---|
| **Read-only / plan mode** | Agent may read files and propose a plan, but cannot write or execute anything | "Show me what you'd change" before touching disk |
| **Edit with approval** | Agent can propose file edits; human approves/denies each one (or a batch) before it's written | Default posture for most tools on first use |
| **Auto-approve safe reads / edits** | Read-only tools (grep, cat, ls) run without prompting; edits and shell commands still gate | Common "sane default" — friction only on things that mutate state |
| **Auto-approve within a sandbox** | Shell commands run freely inside an isolated container/VM with no access to the real filesystem or network | Codex CLI's sandboxed execution model is a good example of this pattern |
| **Full autonomy / unattended** | Agent edits, runs shell commands, commits, and iterates without per-action approval | CI-style or long-running background agent tasks; highest risk, reserved for trusted, scoped tasks |

Why this matters:

- **Blast radius vs. speed** is the core trade-off. Approving every single edit is safe but slow; full autonomy is fast but means a hallucinated or misunderstood instruction can silently do the wrong thing to a real codebase.
- **Destructive shell commands deserve stricter gates than file edits.** A bad file edit is usually recoverable from git; `rm -rf`, `git push --force`, or a database migration may not be.
- **Sandboxing changes the calculus entirely.** If "full autonomy" only ever touches a disposable container, the risk of an unattended agent drops enormously — this is why sandbox-first designs can responsibly default to less human-in-the-loop friction than designs that operate directly on a developer's real machine.
- **Trust is earned per-task, not per-tool.** The same agent might reasonably run unattended on "fix this typo" and require approval on "refactor the auth module." Mature tools let autonomy be configured per session, per directory, or per command pattern rather than as one global on/off switch.

When evaluating any CLI coding agent, this is one of the first things worth checking: **what's auto-approved by default, what always gates, and can the gate policy be configured** to match how much you trust it with a given task.

## 5. Where This Fits in Phase 11

This notebook deliberately stayed at the **landscape level**: what the category is, why the loop looks the way it does across vendors, and what the permission spectrum trades off — reinforced with a toy agent built from primitives (`helpers.get_llm` + a file diff), not any specific product's SDK.

The rest of Phase 11 goes deep on the two ends this notebook intentionally left as surface-level comparisons:

- **`Claude_Code/`** — operator mastery of the real, production Claude Code tool: its actual permission model, hooks, subagents, MCP integration, and daily workflow — the specific implementation of everything Sections 2 and 4 described in the abstract.
- **`Claude_API_and_Agent_SDK/`** — how to build a *real* agent (not a toy) on the Claude API / Agent SDK, including production-grade tool-use loops, context management, and the same permission/autonomy concerns as Section 4, but implemented properly rather than sketched.

Read this notebook first if you're orienting yourself in the coding-agent category broadly; go to those two for depth on Anthropic's specific tools.

## Summary

**Key takeaways:**

- A **CLI coding agent** differs from a chat-based assistant in three concrete ways: direct filesystem access, direct shell access, and an iterative loop where it observes the result of its own actions rather than stopping after one suggestion.
- The landscape (Claude Code, GitHub Copilot CLI/agent mode, Cursor's agent mode, Aider, OpenAI Codex CLI, and others) varies in vendor, model, and UI, but every entrant implements the same underlying **agentic loop**: context gathering \u2192 planning \u2192 tool call \u2192 observation \u2192 repeat, interrupted by permission gates.
- We built that loop's core primitive from scratch — read a file, ask an LLM to edit it via `helpers.get_llm`, apply the edit, observe a diff — deliberately without wrapping any real product, to make the mechanics tangible rather than theoretical.
- Real tools add what our toy agent skipped: self-verification, running tests/linters as part of the loop, multi-file/multi-step planning, and — critically — a **permission/autonomy spectrum** (read-only \u2192 approve-each-edit \u2192 auto-approve-safe-actions \u2192 sandboxed autonomy \u2192 full unattended autonomy) that trades speed against blast radius.
- This notebook is the **landscape entry point** for Phase 11; `Claude_Code/` and `Claude_API_and_Agent_SDK/` are where the deep, product-specific implementations of these same ideas live.